In [ ]:
import os

os.environ["POLARS_MAX_THREADS"] = "4"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

from pathlib import Path
from datetime import datetime, timezone
import gc
import gzip
import re

import numpy as np
import polars as pl
import lightgbm as lgb
from catboost import CatBoostClassifier

TIMESTAMP = "timestamp_ns"

data_dir = Path("../data/raw/data")
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

In [ ]:
def load_targets(data_dir, split):
    data_dir = Path(data_dir)
    targets = pl.read_parquet(data_dir / split / "target_data_solusdt.parquet")
    if split == "train":
        labels = pl.read_parquet(data_dir / "train" / "target_solusdt.parquet")
        assert len(targets) == len(labels)
        if "timestamp_ns" in labels.columns:
            assert targets["timestamp_ns"].equals(labels["timestamp_ns"])
        targets = targets.with_columns(labels["target"], labels["weight"])
    targets = targets.with_row_index("row_id")
    return targets.with_columns(pl.col("row_id").cast(pl.Int64))

In [ ]:
def strict_previous_book(targets, book):
    book = book.rename({"timestamp_ns": "_source_timestamp_ns"})
    book = book.sort("_source_timestamp_ns", maintain_order=True)
    ordered = targets.with_row_index("_original_order")
    ordered = ordered.sort("timestamp_ns", maintain_order=True)
    result = ordered.join_asof(
        book,
        left_on="timestamp_ns",
        right_on="_source_timestamp_ns",
        strategy="backward",
        allow_exact_matches=False,
    )
    return result.sort("_original_order").drop("_original_order")

In [ ]:
def snapshot_features(targets, book, full_features):
    aligned = strict_previous_book(targets, book)
    bid = pl.col("bids[0].price")
    ask = pl.col("asks[0].price")
    bid_amount = pl.col("bids[0].amount")
    ask_amount = pl.col("asks[0].amount")
    mid = (bid.cast(pl.Float64) + ask.cast(pl.Float64)) / 2

    result = aligned.select(
        "row_id",
        "_source_timestamp_ns",
        bid.alias("l1_bid"),
        ask.alias("l1_ask"),
        (ask - bid).alias("l1_spread"),
        (2e4 * (ask - bid) / (ask + bid)).alias("l1_spread_bps"),
        ((bid_amount - ask_amount) / (bid_amount + ask_amount)).alias("l1_imbalance"),
        bid_amount.log1p().alias("l1_log_bid_amount"),
        ask_amount.log1p().alias("l1_log_ask_amount"),
        ((pl.col("timestamp_ns") - pl.col("_source_timestamp_ns")) / 1e6).alias(
            "l1_age_ms"
        ),
        pl.col("_source_timestamp_ns").is_null().cast(pl.Float32).alias("l1_missing"),
    )

    bid_depth = pl.sum_horizontal(
        [pl.col(f"bids[{level}].amount") for level in range(5)],
        ignore_nulls=False,
    )
    ask_depth = pl.sum_horizontal(
        [pl.col(f"asks[{level}].amount") for level in range(5)],
        ignore_nulls=False,
    )
    result = result.hstack(
        aligned.select(
            ((bid_depth - ask_depth) / (bid_depth + ask_depth)).alias(
                "depth_imbalance"
            ),
            (bid_depth + ask_depth).log1p().alias("depth_log_amount"),
        )
    )

    for count in (2, 5, 10, 20):
        bid_depth = pl.sum_horizontal(
            [
                pl.col(f"bids[{level}].amount").cast(pl.Float64)
                for level in range(count)
            ],
            ignore_nulls=False,
        )
        ask_depth = pl.sum_horizontal(
            [
                pl.col(f"asks[{level}].amount").cast(pl.Float64)
                for level in range(count)
            ],
            ignore_nulls=False,
        )
        last_bid = pl.col(f"bids[{count - 1}].price").cast(pl.Float64)
        last_ask = pl.col(f"asks[{count - 1}].price").cast(pl.Float64)
        result = result.hstack(
            aligned.select(
                ((bid_depth - ask_depth) / (bid_depth + ask_depth)).alias(
                    f"regime_imbalance_{count}"
                ),
                (bid_depth + ask_depth).log1p().alias(f"regime_log_depth_{count}"),
                (1e4 * (mid - last_bid) / mid).alias(
                    f"regime_bid_distance_{count}_bps"
                ),
                (1e4 * (last_ask - mid) / mid).alias(
                    f"regime_ask_distance_{count}_bps"
                ),
            )
        )

    result = result.hstack(
        aligned.select(
            pl.col("side").cast(pl.Float32).alias("maker_side"),
            pl.col("price").alias("maker_price"),
            pl.col("amount").log1p().alias("maker_log_amount"),
        )
    )

    if full_features:
        bid = bid.cast(pl.Float64)
        ask = ask.cast(pl.Float64)
        bid_amount = bid_amount.cast(pl.Float64)
        ask_amount = ask_amount.cast(pl.Float64)
        side = pl.col("side").cast(pl.Float64)
        price = pl.col("price").cast(pl.Float64)
        spread = ask - bid
        own_quote = pl.when(side == 1).then(bid).otherwise(ask)
        weighted_mid = (
            pl.when(bid_amount + ask_amount > 0)
            .then((bid_amount * ask + ask_amount * bid) / (bid_amount + ask_amount))
            .otherwise(None)
        )
        result = result.hstack(
            aligned.select(
                (1e4 * side * (mid - price) / mid).alias("execution_mid_edge_bps"),
                (1e4 * side * (own_quote - price) / mid).alias(
                    "execution_quote_edge_bps"
                ),
                (1e4 * side * (weighted_mid - price) / mid).alias(
                    "execution_weighted_mid_edge_bps"
                ),
                pl.when(spread > 0)
                .then(side * (mid - price) / spread)
                .otherwise(None)
                .alias("execution_edge_spreads"),
                (
                    5000
                    * spread
                    / mid
                    * (bid_amount - ask_amount)
                    / (bid_amount + ask_amount)
                ).alias("weighted_mid_bias_bps"),
            )
        )

        bid_depth = pl.sum_horizontal(
            [pl.col(f"bids[{level}].amount").cast(pl.Float64) for level in range(20)],
            ignore_nulls=False,
        )
        ask_depth = pl.sum_horizontal(
            [pl.col(f"asks[{level}].amount").cast(pl.Float64) for level in range(20)],
            ignore_nulls=False,
        )
        bid_value = pl.sum_horizontal(
            [
                pl.col(f"bids[{level}].price").cast(pl.Float64)
                * pl.col(f"bids[{level}].amount").cast(pl.Float64)
                for level in range(20)
            ],
            ignore_nulls=False,
        )
        ask_value = pl.sum_horizontal(
            [
                pl.col(f"asks[{level}].price").cast(pl.Float64)
                * pl.col(f"asks[{level}].amount").cast(pl.Float64)
                for level in range(20)
            ],
            ignore_nulls=False,
        )
        bid_vwap = bid_value / bid_depth
        ask_vwap = ask_value / ask_depth
        result = result.hstack(
            aligned.select(
                (1e4 * (mid - bid_vwap) / mid).alias("shape_bid_cost_bps"),
                (1e4 * (ask_vwap - mid) / mid).alias("shape_ask_cost_bps"),
                (1e4 * (ask_vwap + bid_vwap - 2 * mid) / mid).alias(
                    "shape_cost_skew_bps"
                ),
                (1e4 * (bid - pl.col("bids[19].price").cast(pl.Float64)) / mid).alias(
                    "shape_bid_span_bps"
                ),
                (1e4 * (pl.col("asks[19].price").cast(pl.Float64) - ask) / mid).alias(
                    "shape_ask_span_bps"
                ),
                (bid_amount / bid_depth).alias("shape_top_bid_fraction"),
                (ask_amount / ask_depth).alias("shape_top_ask_fraction"),
            )
        )

        own_queue = pl.when(side == 1).then(bid_amount).otherwise(ask_amount)
        opposite_queue = pl.when(side == 1).then(ask_amount).otherwise(bid_amount)
        amount = pl.col("amount").cast(pl.Float64)
        result = result.hstack(
            aligned.select(
                own_queue.log1p().alias("maker_own_log_queue"),
                opposite_queue.log1p().alias("maker_opposite_log_queue"),
                (amount.log1p() - own_queue.log1p()).alias(
                    "maker_log_size_vs_own_queue"
                ),
                (amount.log1p() - opposite_queue.log1p()).alias(
                    "maker_log_size_vs_opposite_queue"
                ),
            )
        )
    return result

In [ ]:
def orient_features(features):
    columns = []
    for name in features.columns:
        if name in (
            "l1_imbalance",
            "depth_imbalance",
            "weighted_mid_bias_bps",
            "shape_cost_skew_bps",
        ):
            columns.append(name)
        elif name.startswith(
            (
                "regime_imbalance_",
                "flow_return_bps_",
                "queue_lag_imbalance_",
                "queue_change_imbalance_",
                "long_return_bps_",
                "event_return_bps_",
                "event_imbalance_",
                "grid_imbalance_",
            )
        ):
            columns.append(name)
        elif re.fullmatch(r"flow_l\d+_\d+ms", name):
            columns.append(name)
        elif name.startswith("embedding_") and (
            "_return_bps_" in name or "_bias_bps" in name
        ):
            columns.append(name)
    for name in columns:
        features = features.with_columns(
            (pl.col(name) * pl.col("maker_side"))
            .cast(pl.Float32)
            .alias("maker_signed_" + name)
        )
    return features

In [ ]:
def embedding_values(targets, history, instrument):
    history = history.sort(TIMESTAMP, maintain_order=True)
    times = history[TIMESTAMP].to_numpy()
    target_times = targets[TIMESTAMP].to_numpy()
    mid = history["mid_price"].to_numpy().astype(np.float64)
    latest = np.searchsorted(times, target_times, side="left") - 1
    available = latest >= 0
    prefix = "embedding_" + instrument.lower() + "_"
    source = np.zeros(len(targets), dtype=np.int64)
    age = np.full(len(targets), np.nan)
    if len(times):
        source[available] = times[latest[available]]
        age[available] = (target_times[available] - source[available]) / 1000000.0
    output = pl.DataFrame().with_columns(
        pl.Series(
            "_source_" + instrument.lower() + "_timestamp_ns",
            pl.Series(source).set(pl.Series(~available), None),
        ),
        pl.Series(prefix + "age_ms", age),
    )
    complete = available.copy()
    for window in (1000, 5000, 30000):
        lag = np.searchsorted(times, target_times - window * 1000000, side="left") - 1
        valid = available & (lag >= 0)
        complete &= valid
        values = np.full(len(targets), np.nan)
        if len(times):
            values[valid] = 10000.0 * (mid[latest[valid]] / mid[lag[valid]] - 1)
        output = output.with_columns(
            pl.Series(prefix + f"return_bps_{window}ms", values)
        )
    output = output.with_columns(
        pl.Series(prefix + "history_missing", (~complete).astype(np.float64))
    )
    for level in (1, 3, 5, 10):
        values = np.full(len(targets), np.nan)
        vwap = history[f"vwap_{level}_lvl"].to_numpy().astype(np.float64)
        if len(times):
            values[available] = 10000.0 * (
                vwap[latest[available]] / mid[latest[available]] - 1
            )
        output = output.with_columns(
            pl.Series(prefix + f"vwap_{level}_bias_bps", values)
        )
    frame = output
    return frame.with_columns(
        [pl.col(c).cast(pl.Float32) for c in feature_names(frame)]
    )

In [ ]:
def quote_flow(targets, book):
    book = book.sort(TIMESTAMP, maintain_order=True)
    times = book[TIMESTAMP].to_numpy()
    target_times = targets[TIMESTAMP].to_numpy()
    ends = np.searchsorted(times, target_times, side="left")
    latest = np.maximum(ends - 1, 0)
    output = pl.DataFrame()
    levels = 3
    flows = []
    depths = []
    for level in range(levels):
        bid = book[f"bids[{level}].price"].to_numpy().astype(np.float64)
        ask = book[f"asks[{level}].price"].to_numpy().astype(np.float64)
        bid_amount = book[f"bids[{level}].amount"].to_numpy().astype(np.float64)
        ask_amount = book[f"asks[{level}].amount"].to_numpy().astype(np.float64)
        event = np.zeros(len(book), dtype=np.float64)
        event[1:] = (
            (bid[1:] >= bid[:-1]) * bid_amount[1:]
            - (bid[1:] <= bid[:-1]) * bid_amount[:-1]
            - (ask[1:] <= ask[:-1]) * ask_amount[1:]
            + (ask[1:] >= ask[:-1]) * ask_amount[:-1]
        )
        flows.append(np.concatenate(([0.0], np.cumsum(event))))
        depths.append(bid_amount + ask_amount)
    mid = (
        book["bids[0].price"].to_numpy().astype(np.float64)
        + book["asks[0].price"].to_numpy().astype(np.float64)
    ) / 2
    for window in (100, 1000, 5000, 30000):
        begins = np.searchsorted(times, target_times - window * 1000000, side="left")
        valid = (begins > 0) & (ends > 0)
        suffix = str(window) + "ms"
        for level in range(levels):
            values = np.full(len(targets), np.nan)
            if len(book):
                depth = depths[level][latest]
                np.divide(
                    flows[level][ends] - flows[level][begins],
                    depth,
                    out=values,
                    where=valid & (depth > 0),
                )
            output = output.with_columns(
                pl.Series(f"flow_l{level + 1}_{suffix}", values)
            )
        returns = np.full(len(targets), np.nan)
        if len(book):
            before = np.maximum(begins - 1, 0)
            np.divide(
                10000.0 * (mid[latest] - mid[before]),
                mid[before],
                out=returns,
                where=valid & (mid[before] > 0),
            )
        output = output.with_columns(pl.Series(f"flow_return_bps_{suffix}", returns))
        output = output.with_columns(
            pl.Series(f"flow_abs_return_bps_{suffix}", np.abs(returns))
        )
        output = output.with_columns(
            pl.Series(f"flow_log_updates_{suffix}", np.log1p(ends - begins))
        )
        output = output.with_columns(
            pl.Series(f"flow_history_missing_{suffix}", (~valid).astype(np.float64))
        )
    return output

In [ ]:
def queue_dynamics(targets, book):
    book = book.sort(TIMESTAMP, maintain_order=True)
    times = book[TIMESTAMP].to_numpy()
    target_times = targets[TIMESTAMP].to_numpy()
    latest = np.searchsorted(times, target_times, side="left") - 1
    bid_amount = book["bids[0].amount"].to_numpy().astype(np.float64)
    ask_amount = book["asks[0].amount"].to_numpy().astype(np.float64)
    imbalance = np.full(len(book), np.nan)
    np.divide(
        bid_amount - ask_amount,
        bid_amount + ask_amount,
        out=imbalance,
        where=bid_amount + ask_amount > 0,
    )
    output = pl.DataFrame()
    for window in (100, 1000, 5000, 30000):
        lag = np.searchsorted(times, target_times - window * 1000000, side="left") - 1
        valid = (latest >= 0) & (lag >= 0)
        lagged = np.full(len(targets), np.nan)
        change = np.full(len(targets), np.nan)
        if len(book):
            lagged[valid] = imbalance[lag[valid]]
            change[valid] = imbalance[latest[valid]] - lagged[valid]
        suffix = str(window) + "ms"
        output = output.with_columns(pl.Series("queue_lag_imbalance_" + suffix, lagged))
        output = output.with_columns(
            pl.Series("queue_change_imbalance_" + suffix, change)
        )
        output = output.with_columns(
            pl.Series("queue_history_missing_" + suffix, (~valid).astype(np.float64))
        )
    return output

In [ ]:
def liquidity_regime(targets, book):
    book = book.sort(TIMESTAMP, maintain_order=True)
    times = book[TIMESTAMP].to_numpy()
    target_times = targets[TIMESTAMP].to_numpy()
    bid = book["bids[0].price"].to_numpy().astype(np.float64)
    ask = book["asks[0].price"].to_numpy().astype(np.float64)
    mid = (bid + ask) / 2
    returns_bps = np.zeros(len(book), dtype=np.float64)
    if len(book) > 1:
        np.divide(
            10000.0 * (mid[1:] - mid[:-1]),
            mid[:-1],
            out=returns_bps[1:],
            where=mid[:-1] > 0,
        )
    spread_bps = np.full(len(book), np.nan)
    np.divide(10000.0 * (ask - bid), mid, out=spread_bps, where=mid > 0)
    variation_prefix = np.concatenate(([0.0], np.cumsum(returns_bps**2)))
    spread_prefix = np.concatenate(([0.0], np.cumsum(spread_bps)))
    ends = np.searchsorted(times, target_times, side="left")
    output = pl.DataFrame()
    for window in (1000, 5000, 30000):
        begins = np.searchsorted(times, target_times - window * 1000000, side="left")
        valid = (begins > 0) & (ends > 0)
        volatility = np.full(len(targets), np.nan)
        volatility[valid] = np.sqrt(
            np.maximum(
                variation_prefix[ends[valid]] - variation_prefix[begins[valid]], 0
            )
        )
        mean_spread = np.full(len(targets), np.nan)
        np.divide(
            spread_prefix[ends] - spread_prefix[begins],
            ends - begins,
            out=mean_spread,
            where=valid & (ends > begins),
        )
        output = output.with_columns(
            pl.Series(f"regime_volatility_{window}ms", volatility)
        )
        output = output.with_columns(
            pl.Series(f"regime_mean_spread_bps_{window}ms", mean_spread)
        )
    return output

In [ ]:
def state(targets, book):
    book = book.sort(TIMESTAMP, maintain_order=True)
    times = book[TIMESTAMP].to_numpy()
    target_times = targets[TIMESTAMP].to_numpy()
    bid = book["bids[0].price"].to_numpy().astype(np.float64)
    ask = book["asks[0].price"].to_numpy().astype(np.float64)
    bid_amount = book["bids[0].amount"].to_numpy().astype(np.float64)
    ask_amount = book["asks[0].amount"].to_numpy().astype(np.float64)
    ends = np.searchsorted(times, target_times, side="left")
    return (book, times, target_times, bid, ask, bid_amount, ask_amount, ends)

In [ ]:
def event_history(targets, book):
    book, times, target_times, bid, ask, bid_amount, ask_amount, ends = state(
        targets, book
    )
    mid = (bid + ask) / 2
    imbalance = np.full(len(book), np.nan)
    np.divide(
        bid_amount - ask_amount,
        bid_amount + ask_amount,
        out=imbalance,
        where=bid_amount + ask_amount > 0,
    )
    latest = ends - 1
    output = pl.DataFrame()
    for lag in (1, 4, 16, 64):
        previous = latest - lag
        valid = previous >= 0
        returns = np.full(len(targets), np.nan)
        lagged_imbalance = np.full(len(targets), np.nan)
        change = np.full(len(targets), np.nan)
        duration = np.full(len(targets), np.nan)
        source = np.zeros(len(targets), dtype=np.int64)
        returns[valid] = 10000.0 * (mid[latest[valid]] / mid[previous[valid]] - 1)
        lagged_imbalance[valid] = imbalance[previous[valid]]
        change[valid] = imbalance[latest[valid]] - lagged_imbalance[valid]
        duration[valid] = (times[latest[valid]] - times[previous[valid]]) / 1000000.0
        source[valid] = times[previous[valid]]
        output = output.with_columns(
            pl.Series(f"event_return_bps_{lag}", returns),
            pl.Series(f"event_imbalance_{lag}", lagged_imbalance),
            pl.Series(f"event_imbalance_change_{lag}", change),
            pl.Series(f"event_duration_ms_{lag}", duration),
            pl.Series(
                f"_source_event_{lag}_timestamp_ns",
                pl.Series(source).set(pl.Series(~valid), None),
            ),
        )
    return output

In [ ]:
def side_flow(targets, book):
    book, times, target_times, bid, ask, bid_amount, ask_amount, ends = state(
        targets, book
    )
    bid_event = np.zeros(len(book))
    ask_event = np.zeros(len(book))
    bid_event[1:] = (bid[1:] >= bid[:-1]) * bid_amount[1:] - (
        bid[1:] <= bid[:-1]
    ) * bid_amount[:-1]
    ask_event[1:] = (ask[1:] <= ask[:-1]) * ask_amount[1:] - (
        ask[1:] >= ask[:-1]
    ) * ask_amount[:-1]
    prefixes = [np.r_[0.0, np.cumsum(event)] for event in (bid_event, ask_event)]
    output = pl.DataFrame()
    for window in (100, 1000, 5000, 30000):
        begins = np.searchsorted(times, target_times - window * 1000000, side="left")
        valid = (begins > 0) & (ends > 0)
        for side, prefix in zip(("bid", "ask"), prefixes):
            values = np.full(len(targets), np.nan)
            if len(book):
                depth = (bid_amount + ask_amount)[np.maximum(ends - 1, 0)]
                np.divide(
                    prefix[ends] - prefix[begins],
                    depth,
                    out=values,
                    where=valid & (depth > 0),
                )
            output = output.with_columns(
                pl.Series(f"side_flow_{side}_{window}ms", values)
            )
    return output

In [ ]:
def long_regime(targets, book):
    book, times, target_times, bid, ask, bid_amount, ask_amount, ends = state(
        targets, book
    )
    mid = (bid + ask) / 2
    returns = np.zeros(len(book))
    returns[1:] = 10000.0 * (mid[1:] - mid[:-1]) / mid[:-1]
    variation = np.r_[0.0, np.cumsum(returns**2)]
    spread = np.r_[0.0, np.cumsum(10000.0 * (ask - bid) / mid)]
    output = pl.DataFrame()
    for window in (60000, 300000):
        begins = np.searchsorted(times, target_times - window * 1000000, side="left")
        valid = (begins > 0) & (ends > 0)
        volatility = np.full(len(targets), np.nan)
        net_return = np.full(len(targets), np.nan)
        mean_spread = np.full(len(targets), np.nan)
        update_count = np.full(len(targets), np.nan)
        volatility[valid] = np.sqrt(
            np.maximum(variation[ends[valid]] - variation[begins[valid]], 0)
        )
        net_return[valid] = (
            10000.0
            * (mid[ends[valid] - 1] - mid[begins[valid] - 1])
            / mid[begins[valid] - 1]
        )
        np.divide(
            spread[ends] - spread[begins],
            ends - begins,
            out=mean_spread,
            where=valid & (ends > begins),
        )
        update_count[valid] = np.log1p(ends[valid] - begins[valid])
        output = output.with_columns(
            pl.Series(f"long_volatility_{window}ms", volatility),
            pl.Series(f"long_return_bps_{window}ms", net_return),
            pl.Series(f"long_mean_spread_bps_{window}ms", mean_spread),
            pl.Series(f"long_log_updates_{window}ms", update_count),
        )
    return output

In [ ]:
def price_grid(targets, book):
    book, times, target_times, bid, ask, bid_amount, ask_amount, ends = state(
        targets, book
    )
    mid = (bid + ask) / 2
    bid_prices = np.column_stack(
        [book[f"bids[{level}].price"].to_numpy() for level in range(20)]
    )
    ask_prices = np.column_stack(
        [book[f"asks[{level}].price"].to_numpy() for level in range(20)]
    )
    bid_amounts = np.column_stack(
        [book[f"bids[{level}].amount"].to_numpy() for level in range(20)]
    )
    ask_amounts = np.column_stack(
        [book[f"asks[{level}].amount"].to_numpy() for level in range(20)]
    )
    bid_distances = 1e4 * (mid[:, None] - bid_prices) / mid[:, None]
    ask_distances = 1e4 * (ask_prices - mid[:, None]) / mid[:, None]
    latest = np.maximum(ends - 1, 0)
    available = ends > 0
    result = pl.DataFrame()
    for radius in (0.1, 0.5, 1.0, 2.0):
        suffix = format(radius, "g").replace(".", "p") + "bps"
        log_bid = np.full(len(targets), np.nan)
        log_ask = np.full(len(targets), np.nan)
        imbalance = np.full(len(targets), np.nan)
        complete = np.zeros(len(targets), dtype=bool)
        if len(book):
            covered = (bid_distances[:, -1] >= radius) & (
                ask_distances[:, -1] >= radius
            )
            complete = available & covered[latest]
            bid_volumes = np.sum(
                np.where(bid_distances <= radius, bid_amounts, 0.0), axis=1
            )
            ask_volumes = np.sum(
                np.where(ask_distances <= radius, ask_amounts, 0.0), axis=1
            )
            bid_volume = bid_volumes[latest]
            ask_volume = ask_volumes[latest]
            log_bid[complete] = np.log1p(bid_volume[complete])
            log_ask[complete] = np.log1p(ask_volume[complete])
            imbalance[complete & (bid_volume + ask_volume == 0)] = 0.0
            np.divide(
                bid_volume - ask_volume,
                bid_volume + ask_volume,
                out=imbalance,
                where=complete & (bid_volume + ask_volume > 0),
            )
        result = result.with_columns(
            pl.Series("grid_log_bid_" + suffix, log_bid),
            pl.Series("grid_log_ask_" + suffix, log_ask),
            pl.Series("grid_imbalance_" + suffix, imbalance),
            pl.Series("grid_complete_" + suffix, complete.astype(float)),
        )
    return result

In [ ]:
def calculate_book_features(targets, book, full_features):
    features = snapshot_features(targets, book, full_features)
    features = features.hstack(quote_flow(targets, book))
    features = features.hstack(queue_dynamics(targets, book))
    features = features.hstack(liquidity_regime(targets, book))
    features = features.hstack(long_regime(targets, book))
    if full_features:
        features = features.hstack(event_history(targets, book))
        features = features.hstack(side_flow(targets, book))
        features = features.hstack(price_grid(targets, book))
    for name in features.columns:
        if name != "row_id" and not name.startswith("_source_"):
            features = features.with_columns(pl.col(name).cast(pl.Float32))
    if full_features:
        features = orient_features(features)
    return features

In [ ]:
def build_embedding(targets, data_dir, instrument, block_rows):
    columns = [
        "timestamp_ns",
        "instrument",
        "mid_price",
        "vwap_1_lvl",
        "vwap_3_lvl",
        "vwap_5_lvl",
        "vwap_10_lvl",
    ]
    train = pl.scan_parquet(data_dir / "train" / "orderbook_embedding.parquet").select(
        columns
    )
    test = pl.scan_parquet(data_dir / "test" / "orderbook_embedding.parquet").select(
        columns
    )
    stream = pl.concat([train, test]).filter(pl.col("instrument") == instrument)
    ordered = targets.sort(["timestamp_ns", "row_id"])
    lookback = 30_000_000_000
    start = ordered["timestamp_ns"][0] - lookback
    carry = (
        stream.filter(pl.col("timestamp_ns") < start)
        .sort("timestamp_ns", maintain_order=True)
        .tail(1)
        .collect(engine="streaming")
    )
    cursor = start
    pieces = []
    for block in ordered.iter_slices(block_rows):
        end = block["timestamp_ns"][-1]
        events = stream.filter(
            (pl.col("timestamp_ns") >= cursor) & (pl.col("timestamp_ns") < end)
        ).collect(engine="streaming")
        history = pl.concat([carry, events]).sort("timestamp_ns", maintain_order=True)
        values = embedding_values(block, history, instrument)
        pieces.append(block.select("row_id").hstack(values))
        boundary = end - lookback
        predecessor = history.filter(pl.col("timestamp_ns") < boundary).tail(1)
        recent = history.filter(pl.col("timestamp_ns") >= boundary)
        carry = pl.concat([predecessor, recent])
        cursor = end
    return (
        targets.select("row_id")
        .join(
            pl.concat(pieces),
            on="row_id",
            how="left",
            maintain_order="left",
            validate="1:1",
        )
        .drop("row_id")
    )

In [ ]:
def build_features(targets, data_dir, full_features, block_rows=100_000):
    data_dir = Path(data_dir)
    columns = ["timestamp_ns"]
    for level in range(20):
        columns.append(f"bids[{level}].price")
        columns.append(f"bids[{level}].amount")
        columns.append(f"asks[{level}].price")
        columns.append(f"asks[{level}].amount")
    train = pl.scan_parquet(data_dir / "train" / "orderbook_solusdt.parquet").select(
        columns
    )
    test = pl.scan_parquet(data_dir / "test" / "orderbook_solusdt.parquet").select(
        columns
    )
    stream = pl.concat([train, test])
    ordered = targets.sort(["timestamp_ns", "row_id"])
    lookback = 300_000_000_000
    carry_count = 1
    if full_features:
        carry_count = 65
    start = ordered["timestamp_ns"][0] - lookback
    carry = (
        stream.filter(pl.col("timestamp_ns") < start)
        .sort("timestamp_ns", maintain_order=True)
        .tail(carry_count)
        .collect(engine="streaming")
    )
    cursor = start
    pieces = []
    for block in ordered.iter_slices(block_rows):
        end = block["timestamp_ns"][-1]
        events = stream.filter(
            (pl.col("timestamp_ns") >= cursor) & (pl.col("timestamp_ns") < end)
        ).collect(engine="streaming")
        history = pl.concat([carry, events]).sort("timestamp_ns", maintain_order=True)
        pieces.append(calculate_book_features(block, history, full_features))
        boundary = end - lookback
        predecessor = history.filter(pl.col("timestamp_ns") < boundary).tail(
            carry_count
        )
        recent = history.filter(pl.col("timestamp_ns") >= boundary)
        carry = pl.concat([predecessor, recent])
        cursor = end
    result = targets.select("row_id").join(
        pl.concat(pieces),
        on="row_id",
        how="left",
        maintain_order="left",
        validate="1:1",
    )
    del pieces
    result = result.hstack(build_embedding(targets, data_dir, "BTCUSDT", block_rows))
    result = result.hstack(build_embedding(targets, data_dir, "ETHUSDT", block_rows))
    if full_features:
        result = orient_features(result)
    return result

In [ ]:
def feature_names(features):
    return [
        name
        for name in features.columns
        if name != "row_id" and not name.startswith("_source_")
    ]

In [ ]:
def feature_matrix(features, columns):
    matrix = features.select(columns).to_numpy().astype(np.float32, copy=False)
    assert not np.isinf(matrix).any()
    return matrix

In [ ]:
def full_feature_columns(features):
    columns = []
    for name in feature_names(features):
        if name.startswith("maker_signed_"):
            continue
        if name in (
            "maker_own_log_queue",
            "maker_opposite_log_queue",
            "maker_log_size_vs_own_queue",
            "maker_log_size_vs_opposite_queue",
        ):
            continue
        columns.append(name)
    return columns

In [ ]:
train = load_targets(data_dir, "train")
train_start = (
    int(datetime(2024, 9, 18, tzinfo=timezone.utc).timestamp()) * 1_000_000_000
)
train_end = (
    int(datetime(2024, 9, 27, 23, 59, tzinfo=timezone.utc).timestamp()) * 1_000_000_000
)
train = train.filter(
    (pl.col("timestamp_ns") >= train_start) & (pl.col("timestamp_ns") < train_end)
)
assert len(train) == 5_788_718

y_train = train["target"].to_numpy()
w_train = train["weight"].to_numpy()
assert np.isin(y_train, [0, 1]).all()
assert np.isfinite(w_train).all()
assert (w_train >= 0).all()
assert w_train.sum() > 0

trained_dir = output_dir / "trained_models"
trained_dir.mkdir(exist_ok=True)
print("Строк для обучения:", len(train))

In [ ]:
features_lgb_base = build_features(train, data_dir, full_features=False)
assert features_lgb_base["row_id"].equals(train["row_id"])
columns_base = feature_names(features_lgb_base)
x_lgb_base = feature_matrix(features_lgb_base, columns_base)
del features_lgb_base
gc.collect()

In [ ]:
model_lgb_base = lgb.LGBMClassifier(
    n_estimators=201,
    learning_rate=0.025,
    num_leaves=127,
    min_child_samples=500,
    max_bin=127,
    reg_lambda=10,
    colsample_bytree=0.85,
    subsample=0.8,
    subsample_freq=1,
    objective="binary",
    random_state=42,
    n_jobs=4,
    device_type="cpu",
    deterministic=True,
    force_col_wise=True,
    verbosity=-1,
)
model_lgb_base.fit(x_lgb_base, y_train, sample_weight=w_train)
model_lgb_base.booster_.save_model(str(trained_dir / "lightgbm_base.txt"))
del x_lgb_base
gc.collect()

In [ ]:
features_full = build_features(train, data_dir, full_features=True)
assert features_full["row_id"].equals(train["row_id"])
columns_full = full_feature_columns(features_full)
columns_relative = feature_names(features_full)
features_path = output_dir / "train_features.parquet"
features_full.select(columns_relative).write_parquet(features_path)
del features_full
gc.collect()

x_lgb_full = (
    pl.read_parquet(features_path, columns=columns_full)
    .to_numpy()
    .astype(np.float32, copy=False)
)
assert not np.isinf(x_lgb_full).any()

In [ ]:
model_lgb_full = lgb.LGBMClassifier(
    n_estimators=290,
    learning_rate=0.025,
    num_leaves=127,
    min_child_samples=500,
    max_bin=127,
    reg_lambda=10,
    colsample_bytree=0.85,
    subsample=0.8,
    subsample_freq=1,
    objective="binary",
    random_state=42,
    n_jobs=4,
    device_type="cpu",
    deterministic=True,
    force_col_wise=True,
    verbosity=-1,
)
model_lgb_full.fit(x_lgb_full, y_train, sample_weight=w_train)
model_lgb_full.booster_.save_model(str(trained_dir / "lightgbm_full.txt"))

In [ ]:
x_cat_deep = x_lgb_full
model_cat_deep = CatBoostClassifier(
    iterations=591,
    depth=8,
    l2_leaf_reg=10,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Logloss",
    boosting_type="Plain",
    bootstrap_type="Bernoulli",
    border_count=127,
    has_time=True,
    random_strength=1,
    subsample=0.8,
    random_seed=42,
    task_type="GPU",
    devices="0",
    thread_count=4,
    gpu_ram_part=0.7,
    pinned_memory_size="67108864",
    allow_writing_files=False,
    verbose=100,
)
model_cat_deep.fit(x_cat_deep, y_train, sample_weight=w_train)
model_cat_deep.save_model(str(trained_dir / "catboost_deep.cbm"))
del x_lgb_full, x_cat_deep
gc.collect()

In [ ]:
x_cat_shallow = (
    pl.read_parquet(features_path, columns=columns_relative)
    .to_numpy()
    .astype(np.float32, copy=False)
)
assert not np.isinf(x_cat_shallow).any()
model_cat_shallow = CatBoostClassifier(
    iterations=1388,
    depth=4,
    l2_leaf_reg=30,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Logloss",
    boosting_type="Plain",
    bootstrap_type="Bernoulli",
    border_count=127,
    has_time=True,
    random_strength=1,
    subsample=0.8,
    random_seed=42,
    task_type="GPU",
    devices="0",
    thread_count=4,
    gpu_ram_part=0.7,
    pinned_memory_size="67108864",
    allow_writing_files=False,
    verbose=100,
)
model_cat_shallow.fit(x_cat_shallow, y_train, sample_weight=w_train)
model_cat_shallow.save_model(str(trained_dir / "catboost_shallow.cbm"))
del x_cat_shallow, train, y_train, w_train
gc.collect()

In [ ]:
model_dir = trained_dir

In [ ]:
test = load_targets(data_dir, "test")
features_test_lgb_base = build_features(test, data_dir, full_features=False)
assert features_test_lgb_base["row_id"].equals(test["row_id"])
columns_base = feature_names(features_test_lgb_base)
x_test_lgb_base = feature_matrix(features_test_lgb_base, columns_base)

model_lgb_base = lgb.Booster(model_file=str(model_dir / "lightgbm_base.txt"))
p_lgb_base = model_lgb_base.predict(x_test_lgb_base, num_iteration=201, num_threads=4)
del features_test_lgb_base, x_test_lgb_base
gc.collect()

In [ ]:
features_test_full = build_features(test, data_dir, full_features=True)
assert features_test_full["row_id"].equals(test["row_id"])
columns_full = full_feature_columns(features_test_full)
columns_relative = feature_names(features_test_full)
x_test_lgb_full = feature_matrix(features_test_full, columns_full)

model_lgb_full = lgb.Booster(model_file=str(model_dir / "lightgbm_full.txt"))
p_lgb_full = model_lgb_full.predict(x_test_lgb_full, num_iteration=290, num_threads=4)

model_cat_deep = CatBoostClassifier()
model_cat_deep.load_model(str(model_dir / "catboost_deep.cbm"))
p_cat_deep = model_cat_deep.predict_proba(x_test_lgb_full, thread_count=4)[:, 1]
del x_test_lgb_full
gc.collect()

In [ ]:
x_test_cat_shallow = feature_matrix(features_test_full, columns_relative)
model_cat_shallow = CatBoostClassifier()
model_cat_shallow.load_model(str(model_dir / "catboost_shallow.cbm"))
p_cat_shallow = model_cat_shallow.predict_proba(x_test_cat_shallow, thread_count=4)[
    :, 1
]
del features_test_full, x_test_cat_shallow
gc.collect()

In [ ]:
def individual_submission(ids, probabilities, path):
    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0).all()
    assert (probabilities <= 1).all()
    frame = pl.DataFrame({"id": ids, "prediction": probabilities}).sort("id")
    frame.write_csv(path, float_precision=10)
    return pl.read_csv(path)


submission_lgb_base = individual_submission(
    test["id"], p_lgb_base, output_dir / "lightgbm_base.csv"
)
submission_lgb_full = individual_submission(
    test["id"], p_lgb_full, output_dir / "lightgbm_full.csv"
)
submission_cat_deep = individual_submission(
    test["id"], p_cat_deep, output_dir / "catboost_deep.csv"
)
submission_cat_shallow = individual_submission(
    test["id"], p_cat_shallow, output_dir / "catboost_shallow.csv"
)

assert submission_lgb_base["id"].equals(submission_cat_shallow["id"])
assert submission_lgb_full["id"].equals(submission_cat_shallow["id"])
assert submission_cat_deep["id"].equals(submission_cat_shallow["id"])

tree_blend_prediction = np.zeros(len(test))
tree_blend_prediction += 0.60 * submission_cat_deep["prediction"].to_numpy()
tree_blend_prediction += 0.20 * submission_lgb_base["prediction"].to_numpy()
tree_blend_prediction += 0.20 * submission_lgb_full["prediction"].to_numpy()
tree_blend = pl.DataFrame(
    {"id": submission_cat_shallow["id"], "prediction": tree_blend_prediction}
)
tree_blend_path = output_dir / "tree_blend.csv"
tree_blend.write_csv(tree_blend_path, float_precision=10)
tree_blend = pl.read_csv(tree_blend_path)

prediction = np.zeros(len(test))
prediction += 0.50 * submission_cat_shallow["prediction"].to_numpy()
prediction += 0.50 * tree_blend["prediction"].to_numpy()
submission = pl.DataFrame(
    {"id": submission_cat_shallow["id"], "prediction": prediction}
)

assert submission.columns == ["id", "prediction"]
assert len(submission) == len(test)
assert submission["id"].n_unique() == len(test)
assert submission["id"].equals(test["id"].sort())
assert np.isfinite(prediction).all()
assert (prediction >= 0).all()
assert (prediction <= 1).all()

submission_path = output_dir / "submission.csv.gz"
submission_path.write_bytes(gzip.compress(submission.write_csv().encode(), mtime=0))
print(submission_path.resolve())
print("Строк:", len(submission))
submission.head()